# Visible Points Depth Analysis

This notebook analyzes visible points and their depth characteristics:
1. Loads metadata from a sequence
2. Given a frame ID, extracts all visible points (excluding newly sampled keypoints at that frame)
3. Computes and visualizes analysis on:
   - How many visible points have valid depth values
   - Depth situation of pixels around visible points
   - **Depth uncertainty (covariance) analysis**

## Execution Order

**IMPORTANT:** Execute cells in this order:

1. **Cell 1**: Imports
2. **Cell 2-3**: Helper Functions
3. **Cell 4-5**: Configuration
4. **Cell 6-11**: ⚠️ **SKIP THESE** - They contain redirect notes (cells were moved)
5. **Cell 12-13**: Load Metadata and Extract Visible Points
6. **Cell 14-15**: Load Frame Data (RGB and Depth) - **Creates `frame_data`**
7. **Cell 16-17**: Analyze Depth Around Visible Points
8. **Cell 18-19**: **Depth Uncertainty Analysis** - Requires `frame_data` from step 6
9. **Cell 20+**: Visualizations and Detailed Analysis

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from typing import Optional, Tuple
from scipy.ndimage import uniform_filter

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from point2pose.io.sources.dataset.datareader import Ho3dReader, YcbineoatReader
from point2pose.data_types.frame import Frame
from point2pose.utils.camera import convert_pixel_to_world

# Use widget backend for interactive 3D plots (requires ipympl: pip install ipympl)
# Falls back to inline if widget is not available
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('matplotlib', 'widget')
        print("Using matplotlib widget backend for interactive 3D plots")
    else:
        import matplotlib
        matplotlib.use('TkAgg')  # Fallback for non-IPython environments
        print("Using TkAgg backend for interactive 3D plots")
except Exception as e:
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic('matplotlib', 'inline')
        print("Using matplotlib inline backend (3D plots may not be fully interactive)")
        print("For full interactivity, install ipympl: pip install ipympl, then restart kernel and use %matplotlib widget")
    except:
        import matplotlib
        matplotlib.use('Agg')
        print("Using non-interactive backend")

## Helper Functions

In [ ]:
def unpack_ragged(name: str, store: dict, dim=-1):
    """Unpack ragged array from metadata storage."""
    try:
        data = store[f"{name}_data"]
        offsets = store[f"{name}_offsets"]
        lengths = store[f"{name}_lengths"]
        out = []
        for off, L in zip(offsets, lengths):
            flat_data = data[off : off + L]
            if dim == 3:
                reshaped_data = flat_data.reshape(-1, 3)
            elif dim == 2:
                reshaped_data = flat_data.reshape(-1, 2)
            else:
                reshaped_data = flat_data
            out.append(reshaped_data)
        return out
    except KeyError:
        return []

def load_metadata(path: str):
    """Load metadata from NPZ file."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} does not exist")
    data = np.load(path, allow_pickle=True)
    return data

def get_frame_visible_points(data: dict, frame_id: int, exclude_new_keypoints: bool = True):
    """
    Extract visible points for a given frame, optionally excluding newly sampled keypoints.
    
    Args:
        data: Loaded metadata dictionary
        frame_id: Frame ID to extract points for
        exclude_new_keypoints: If True, exclude points that were newly sampled at this frame
        
    Returns:
        dict with keys: 'points_2d', 'points_3d', 'track_indices', 'valid_depth', 'visible_mask'
    """
    # Find frame index
    frame_ids = data['frame_id']
    frame_idx = None
    for i, fid in enumerate(frame_ids):
        if fid == frame_id:
            frame_idx = i
            break
    
    if frame_idx is None:
        raise ValueError(f"Frame {frame_id} not found in metadata")
    
    # Load tracked 2D points
    track2d_list = unpack_ragged("track2d", data, dim=2)
    if frame_idx >= len(track2d_list):
        return {
            'points_2d': np.array([]).reshape(0, 2),
            'points_3d': np.array([]).reshape(0, 3),
            'track_indices': np.array([], dtype=int),
            'valid_depth': np.array([], dtype=bool),
            'visible_mask': np.array([], dtype=bool)
        }
    
    points_2d = track2d_list[frame_idx]
    
    # Load visibility mask
    visibles_list = unpack_ragged("visibles", data)
    if frame_idx < len(visibles_list):
        visible_mask = visibles_list[frame_idx].astype(bool)
    else:
        visible_mask = np.ones(len(points_2d), dtype=bool)
    
    # Load valid depth mask
    valid_depth_list = unpack_ragged("valid_depth", data)
    if frame_idx < len(valid_depth_list):
        valid_depth = valid_depth_list[frame_idx].astype(bool)
    else:
        valid_depth = np.ones(len(points_2d), dtype=bool)
    
    # Load 3D points
    track3d_list = unpack_ragged("track3d", data, dim=3)
    if frame_idx < len(track3d_list):
        points_3d = track3d_list[frame_idx]
    else:
        points_3d = np.zeros((len(points_2d), 3))
    
    # Get track indices (assuming they're sequential)
    track_indices = np.arange(len(points_2d))
    
    # Filter to only visible points
    visible_indices = np.where(visible_mask)[0]
    
    # Exclude newly sampled keypoints at this frame if requested
    if exclude_new_keypoints:
        obj_key_point_frames_list = unpack_ragged("obj_key_point_frames", data)
        if frame_idx < len(obj_key_point_frames_list) and len(obj_key_point_frames_list[frame_idx]) > 0:
            # Get frame IDs when each keypoint was sampled
            kp_frame_ids = obj_key_point_frames_list[frame_idx]
            # Points that were NOT newly sampled at this frame
            not_new_at_frame = kp_frame_ids != frame_id
            # Combine with visible mask
            visible_indices = visible_indices[not_new_at_frame[visible_indices]]
    
    return {
        'points_2d': points_2d[visible_indices],
        'points_3d': points_3d[visible_indices],
        'track_indices': track_indices[visible_indices],
        'valid_depth': valid_depth[visible_indices],
        'visible_mask': visible_mask[visible_indices]
    }

## Configuration

In [ ]:
# --- CONFIGURATION ---
FRAME_ID = 227 # Change this to analyze a different frame

# Path to metadata file
results_dir = '/home/justin/code/point-to-pose/results/ho3d_single'
video_name = 'MPM10'
meta_data_path = os.path.join(results_dir, video_name, 'meta_data', 'meta_data.npz')

# Dataset paths (for loading RGB and depth)
ho3d_root = '/home/justin/data/HO3D_V3/'  # Adjust to your HO3D root
video_dir = os.path.join(ho3d_root, 'evaluation', video_name)  # Adjust if needed

# Alternative: If using YCBInEOAT dataset
# ycbineoat_root = '/path/to/ycbineoat'
# video_dir = os.path.join(ycbineoat_root, video_name)

print(f"Loading metadata from: {meta_data_path}")
print(f"Frame ID to analyze: {FRAME_ID}")

In [ ]:
# NOTE: Run the "Depth Uncertainty Analysis" cells (after the depth analysis section) first!

# Detailed uncertainty analysis and comparisons
if ('uncertainty_analysis' in globals() and 'depth_analysis' in globals() and 
    frame_data is not None and uncertainty_analysis is not None and depth_analysis is not None):
    valid_mask = uncertainty_analysis['valid_mask']
    valid_sigma_z = uncertainty_analysis['valid_sigma_z']
    valid_depths = uncertainty_analysis['valid_depths']
    
    # Match indices between uncertainty analysis and depth analysis
    valid_uncertainty_indices = np.where(valid_mask)[0]
    
    # Initialize neighbor statistics variables
    neighbor_stds = None
    neighbor_means = None
    neighbor_valid_ratios = None
    valid_neighbor_mask = np.array([], dtype=bool)
    
    if len(valid_uncertainty_indices) > 0:
        neighbor_stds = depth_analysis['neighbor_std_depths'][valid_uncertainty_indices]
        neighbor_means = depth_analysis['neighbor_mean_depths'][valid_uncertainty_indices]
        neighbor_valid_ratios = depth_analysis['neighbor_valid_ratios'][valid_uncertainty_indices]
        valid_neighbor_mask = np.isfinite(neighbor_stds) & np.isfinite(neighbor_means)
    
    print("\\n" + "="*60)
    print("DETAILED UNCERTAINTY ANALYSIS")
    print("="*60)
    
    # 1. Correlation with depth
    if len(valid_depths) > 1:
        depth_corr = np.corrcoef(valid_depths, valid_sigma_z)[0, 1]
        print(f"\\n1. Correlation with Depth:")
        print(f"   Correlation coefficient: {depth_corr:.4f}")
        if abs(depth_corr) > 0.3:
            print(f"   -> Strong correlation: uncertainty {'increases' if depth_corr > 0 else 'decreases'} with depth")
        else:
            print(f"   -> Weak correlation: uncertainty is relatively independent of depth")
    
    # 2. Correlation with neighbor statistics
    if neighbor_stds is not None and np.sum(valid_neighbor_mask) > 1:
        std_corr = np.corrcoef(neighbor_stds[valid_neighbor_mask], 
                              valid_sigma_z[valid_neighbor_mask])[0, 1]
        mean_corr = np.corrcoef(neighbor_means[valid_neighbor_mask], 
                               valid_sigma_z[valid_neighbor_mask])[0, 1]
        ratio_corr = np.corrcoef(neighbor_valid_ratios[valid_neighbor_mask], 
                                valid_sigma_z[valid_neighbor_mask])[0, 1]
        
        print(f"\\n2. Correlation with Neighbor Statistics:")
        print(f"   With neighbor depth std: {std_corr:.4f}")
        print(f"   With neighbor depth mean: {mean_corr:.4f}")
        print(f"   With neighbor valid ratio: {ratio_corr:.4f}")
    
    # 3. Uncertainty by depth ranges
    print(f"\\n3. Uncertainty Statistics by Depth Ranges:")
    depth_ranges = [(0.05, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 2.0)]
    for d_min, d_max in depth_ranges:
        range_mask = (valid_depths >= d_min) & (valid_depths < d_max)
        if np.any(range_mask):
            range_uncertainty = valid_sigma_z[range_mask]
            print(f"   Depth [{d_min:.2f}, {d_max:.2f})m: "
                  f"mean={np.mean(range_uncertainty):.6f}m, "
                  f"median={np.median(range_uncertainty):.6f}m, "
                  f"count={np.sum(range_mask)}")
    
    # 4. High vs Low uncertainty analysis
    print(f"\\n4. High vs Low Uncertainty Comparison:")
    median_uncertainty = np.median(valid_sigma_z)
    low_uncertainty_mask = valid_sigma_z < median_uncertainty
    high_uncertainty_mask = valid_sigma_z >= median_uncertainty
    
    print(f"   Low uncertainty (< median {median_uncertainty:.6f}m):")
    print(f"     Count: {np.sum(low_uncertainty_mask)}")
    print(f"     Mean depth: {np.mean(valid_depths[low_uncertainty_mask]):.3f}m")
    if neighbor_stds is not None and np.sum(valid_neighbor_mask) > 0:
        # Map low_uncertainty_mask to the valid_neighbor_mask indices
        low_mask_mapped = low_uncertainty_mask[valid_uncertainty_indices][valid_neighbor_mask]
        if np.any(low_mask_mapped):
            low_neighbor_stds = neighbor_stds[valid_neighbor_mask][low_mask_mapped]
            if len(low_neighbor_stds) > 0:
                print(f"     Mean neighbor std: {np.mean(low_neighbor_stds):.6f}m")
    
    print(f"   High uncertainty (>= median {median_uncertainty:.6f}m):")
    print(f"     Count: {np.sum(high_uncertainty_mask)}")
    print(f"     Mean depth: {np.mean(valid_depths[high_uncertainty_mask]):.3f}m")
    if neighbor_stds is not None and np.sum(valid_neighbor_mask) > 0:
        # Map high_uncertainty_mask to the valid_neighbor_mask indices
        high_mask_mapped = high_uncertainty_mask[valid_uncertainty_indices][valid_neighbor_mask]
        if np.any(high_mask_mapped):
            high_neighbor_stds = neighbor_stds[valid_neighbor_mask][high_mask_mapped]
            if len(high_neighbor_stds) > 0:
                print(f"     Mean neighbor std: {np.mean(high_neighbor_stds):.6f}m")
    
    # 5. Uncertainty distribution percentiles
    print(f"\\n5. Uncertainty Distribution Percentiles:")
    percentiles = [5, 10, 25, 50, 75, 90, 95]
    for p in percentiles:
        p_val = np.percentile(valid_sigma_z, p)
        print(f"   P{p}: {p_val:.6f}m")
    
    # 6. Edge detection analysis (uncertainty should be higher at edges)
    print(f"\\n6. Edge vs Non-Edge Analysis:")
    # Use neighbor std as a proxy for edge detection
    if neighbor_stds is not None and np.sum(valid_neighbor_mask) > 0:
        edge_threshold = np.percentile(neighbor_stds[valid_neighbor_mask], 75)
        edge_mask = neighbor_stds[valid_neighbor_mask] > edge_threshold
        
        edge_uncertainty = valid_sigma_z[valid_neighbor_mask][edge_mask]
        non_edge_uncertainty = valid_sigma_z[valid_neighbor_mask][~edge_mask]
        
        if len(edge_uncertainty) > 0 and len(non_edge_uncertainty) > 0:
            print(f"   Edge regions (high neighbor std > {edge_threshold:.6f}m):")
            print(f"     Mean uncertainty: {np.mean(edge_uncertainty):.6f}m")
            print(f"     Count: {len(edge_uncertainty)}")
            print(f"   Non-edge regions:")
            print(f"     Mean uncertainty: {np.mean(non_edge_uncertainty):.6f}m")
            print(f"     Count: {len(non_edge_uncertainty)}")
            print(f"   Ratio (edge/non-edge): {np.mean(edge_uncertainty)/np.mean(non_edge_uncertainty):.2f}x")
    
    print("\\n" + "="*60)
    
else:
    print("Cannot perform detailed uncertainty analysis: required data not available")

## Load Metadata and Extract Visible Points

In [ ]:
# Load metadata
meta_data = load_metadata(meta_data_path)
num_frames = len(meta_data['frame_id'])
print(f"Loaded {num_frames} frames from metadata")

# Extract visible points (excluding newly sampled keypoints at this frame)
visible_points = get_frame_visible_points(meta_data, FRAME_ID, exclude_new_keypoints=True)

print(f"\nVisible points (excluding newly sampled at frame {FRAME_ID}):")
print(f"  Total visible points: {len(visible_points['points_2d'])}")
print(f"  Points with valid depth: {np.sum(visible_points['valid_depth'])}")
print(f"  Points without valid depth: {np.sum(~visible_points['valid_depth'])}")
print(f"  Valid depth percentage: {100 * np.mean(visible_points['valid_depth']):.2f}%")

## Load Frame Data (RGB and Depth)

In [ ]:
# Load frame data
reader = None
frame_data = None

# Try to create reader and load frame
if os.path.exists(video_dir):
    try:
        # Try HO3D first
        if os.path.exists(os.path.join(ho3d_root, 'models')):
            reader = Ho3dReader(video_dir, ho3d_root)
            print(f"Created Ho3dReader with {len(reader)} frames")
        else:
            # Try YCBInEOAT
            reader = YcbineoatReader(video_dir)
            print(f"Created YcbineoatReader with {len(reader)} frames")
        
        # Load frame
        if FRAME_ID < len(reader):
            if isinstance(reader, Ho3dReader):
                rgb = cv2.cvtColor(cv2.imread(reader.color_files[FRAME_ID]), cv2.COLOR_BGR2RGB)
                depth = reader.get_depth(FRAME_ID)
                mask = reader.get_mask(FRAME_ID)
                H, W = rgb.shape[:2]
                mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
                intrinsics = reader.K
                depth_factor = 1.0
            else:  # YCBInEOAT
                rgb = reader.get_color(FRAME_ID)
                depth = reader.get_depth(FRAME_ID)
                mask = reader.get_mask(FRAME_ID)
                H, W = rgb.shape[:2]
                intrinsics = reader.K
                depth_factor = 1.0
            
            frame_data = {
                'rgb': rgb,
                'depth': depth,
                'mask': mask,
                'intrinsics': intrinsics,
                'depth_factor': depth_factor,
                'H': H,
                'W': W
            }
            print(f"Loaded frame {FRAME_ID}: RGB shape {rgb.shape}, Depth shape {depth.shape}")
        else:
            print(f"Warning: Frame {FRAME_ID} not available in reader (max: {len(reader)-1})")
    except Exception as e:
        print(f"Warning: Could not load frame data: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"Warning: Video directory not found: {video_dir}")
    print("Frame visualization will be skipped, but depth analysis can still be done with metadata.")

## Analyze Depth Around Visible Points

In [ ]:
def analyze_depth_around_points(
    points_2d: np.ndarray,
    depth_image: np.ndarray,
    window_size: int = 5,
    min_depth: float = 0.05,
    max_depth: float = 2.0
) -> dict:
    """
    Analyze depth situation around each point.
    
    Args:
        points_2d: (N, 2) pixel coordinates
        depth_image: (H, W) depth image
        window_size: Size of neighborhood window (odd integer)
        min_depth: Minimum valid depth
        max_depth: Maximum valid depth
        
    Returns:
        dict with analysis results
    """
    H, W = depth_image.shape
    half = window_size // 2
    N = len(points_2d)
    
    results = {
        'point_depths': np.full(N, np.nan),
        'point_valid': np.zeros(N, dtype=bool),
        'neighbor_valid_counts': np.zeros(N, dtype=int),
        'neighbor_mean_depths': np.full(N, np.nan),
        'neighbor_std_depths': np.full(N, np.nan),
        'neighbor_valid_ratios': np.zeros(N),
    }
    
    for i, (x, y) in enumerate(points_2d):
        x_int = int(np.round(x))
        y_int = int(np.round(y))
        
        # Check if point is in bounds
        if 0 <= x_int < W and 0 <= y_int < H:
            # Get depth at point
            depth_at_point = depth_image[y_int, x_int]
            if np.isfinite(depth_at_point) and min_depth <= depth_at_point <= max_depth:
                results['point_depths'][i] = depth_at_point
                results['point_valid'][i] = True
            
            # Analyze neighborhood
            x0 = max(0, x_int - half)
            x1 = min(W, x_int + half + 1)
            y0 = max(0, y_int - half)
            y1 = min(H, y_int + half + 1)
            
            neighborhood = depth_image[y0:y1, x0:x1]
            valid_mask = np.isfinite(neighborhood) & (neighborhood >= min_depth) & (neighborhood <= max_depth)
            
            results['neighbor_valid_counts'][i] = np.sum(valid_mask)
            results['neighbor_valid_ratios'][i] = np.sum(valid_mask) / neighborhood.size
            
            if np.any(valid_mask):
                valid_depths = neighborhood[valid_mask]
                results['neighbor_mean_depths'][i] = np.mean(valid_depths)
                results['neighbor_std_depths'][i] = np.std(valid_depths)
    
    return results

# Analyze depth around visible points
if frame_data is not None:
    depth_analysis = analyze_depth_around_points(
        visible_points['points_2d'],
        frame_data['depth'],
        window_size=5,
        min_depth=0.05,
        max_depth=2.0
    )
    
    print("\nDepth Analysis Results:")
    print(f"  Points with valid depth at location: {np.sum(depth_analysis['point_valid'])}")
    print(f"  Points with valid depth in neighborhood: {np.sum(depth_analysis['neighbor_valid_counts'] > 0)}")
    print(f"  Mean valid neighbors per point: {np.mean(depth_analysis['neighbor_valid_counts']):.2f}")
    print(f"  Mean neighbor valid ratio: {np.mean(depth_analysis['neighbor_valid_ratios']):.2f}")
    
    # Compare with metadata valid_depth
    if len(visible_points['valid_depth']) > 0:
        metadata_valid = visible_points['valid_depth']
        depth_image_valid = depth_analysis['point_valid']
        if len(metadata_valid) == len(depth_image_valid):
            agreement = np.sum(metadata_valid == depth_image_valid)
            print(f"  Agreement with metadata valid_depth: {agreement}/{len(metadata_valid)} ({100*agreement/len(metadata_valid):.2f}%)")
else:
    print("Cannot analyze depth around points: frame data not loaded")
    depth_analysis = None

## Depth Uncertainty Analysis

Analyze depth uncertainty (covariance) estimated by convert_pixel_to_world with compute_depth_uncertainty=True.

In [ ]:
# Compute depth uncertainty for visible points
if frame_data is not None:
    points_2d = visible_points['points_2d']
    
    # Compute world points with depth uncertainty
    world_pts, valid_mask, sigma_z = convert_pixel_to_world(
        pixel=points_2d,
        depth_image=frame_data['depth'],
        cam_intrinsics=frame_data['intrinsics'],
        depth_factor=frame_data['depth_factor'],
        cam2world=np.eye(4),  # Camera frame
        remove_invalid=False,
        min_depth=0.05,
        max_depth=2.0,
        fill_missing_depth=False,
        window_size=5,
        min_neighbors=1,
        compute_depth_uncertainty=True,
        sigma_min=0.002,
        sigma_max=0.05,
        sigma_base_a=0.003,
        sigma_base_b=0.0,
        edge_alpha=5.0,
    )
    
    # Extract valid uncertainty values
    valid_uncertainty_mask = np.isfinite(sigma_z) & valid_mask
    valid_sigma_z = sigma_z[valid_uncertainty_mask]
    valid_points_2d = points_2d[valid_uncertainty_mask]
    valid_depths = world_pts[valid_uncertainty_mask, 2]  # Z coordinate (depth)
    
    print("\\nDepth Uncertainty Analysis:")
    print(f"  Total visible points: {len(points_2d)}")
    print(f"  Points with valid uncertainty: {np.sum(valid_uncertainty_mask)}")
    print(f"  Mean uncertainty (sigma_z): {np.mean(valid_sigma_z):.6f}m")
    print(f"  Std uncertainty: {np.std(valid_sigma_z):.6f}m")
    print(f"  Min uncertainty: {np.min(valid_sigma_z):.6f}m")
    print(f"  Max uncertainty: {np.max(valid_sigma_z):.6f}m")
    print(f"  Median uncertainty: {np.median(valid_sigma_z):.6f}m")
    
    # Store for later use
    uncertainty_analysis = {
        'sigma_z': sigma_z,
        'valid_mask': valid_uncertainty_mask,
        'valid_sigma_z': valid_sigma_z,
        'valid_points_2d': valid_points_2d,
        'valid_depths': valid_depths,
        'world_pts': world_pts
    }
else:
    print("Cannot compute depth uncertainty: frame data not loaded")
    uncertainty_analysis = None

## Depth Uncertainty Visualizations

## Visualizations

In [ ]:
# Create comprehensive visualization
if frame_data is not None and depth_analysis is not None:
    fig = plt.figure(figsize=(20, 12))
    
    # 1. RGB image with visible points
    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(frame_data['rgb'])
    points_2d = visible_points['points_2d']
    valid_mask = depth_analysis['point_valid']
    invalid_mask = ~valid_mask
    
    # Plot valid points in green, invalid in red
    if np.any(valid_mask):
        ax1.scatter(points_2d[valid_mask, 0], points_2d[valid_mask, 1], 
                   c='green', s=20, alpha=0.6, label=f'Valid depth ({np.sum(valid_mask)})')
    if np.any(invalid_mask):
        ax1.scatter(points_2d[invalid_mask, 0], points_2d[invalid_mask, 1], 
                   c='red', s=20, alpha=0.6, label=f'Invalid depth ({np.sum(invalid_mask)})')
    ax1.set_title(f'Visible Points on RGB (Frame {FRAME_ID})')
    ax1.legend()
    ax1.axis('off')
    
    # 2. Depth image with visible points
    ax2 = plt.subplot(2, 3, 2)
    depth_vis = frame_data['depth'].copy()
    depth_vis[depth_vis > 2.0] = 2.0  # Clamp for visualization
    depth_vis[depth_vis < 0.05] = 0.05
    ax2.imshow(depth_vis, cmap='jet', vmin=0.05, vmax=2.0)
    if np.any(valid_mask):
        ax2.scatter(points_2d[valid_mask, 0], points_2d[valid_mask, 1], 
                   c='green', s=20, alpha=0.8, edgecolors='white', linewidths=0.5)
    if np.any(invalid_mask):
        ax2.scatter(points_2d[invalid_mask, 0], points_2d[invalid_mask, 1], 
                   c='red', s=20, alpha=0.8, edgecolors='white', linewidths=0.5)
    ax2.set_title('Depth Map with Visible Points')
    ax2.axis('off')
    plt.colorbar(ax2.images[0], ax=ax2, label='Depth (m)')
    
    # 3. Valid depth statistics
    ax3 = plt.subplot(2, 3, 3)
    categories = ['Valid\nDepth', 'Invalid\nDepth']
    counts = [np.sum(valid_mask), np.sum(invalid_mask)]
    colors = ['green', 'red']
    bars = ax3.bar(categories, counts, color=colors, alpha=0.7)
    ax3.set_ylabel('Number of Points')
    ax3.set_title('Depth Validity Distribution')
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\\n({100*count/len(valid_mask):.1f}%)',
                ha='center', va='bottom')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Neighbor valid ratio distribution
    ax4 = plt.subplot(2, 3, 4)
    neighbor_valid_ratios = depth_analysis['neighbor_valid_ratios']
    ax4.hist(neighbor_valid_ratios, bins=20, alpha=0.7, color='blue', edgecolor='black')
    ax4.axvline(np.mean(neighbor_valid_ratios), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(neighbor_valid_ratios):.2%}')
    ax4.set_xlabel('Neighbor Valid Ratio')
    ax4.set_ylabel('Number of Points')
    ax4.set_title('Distribution of Valid Depth Ratio in Neighborhood')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Depth values at points
    ax5 = plt.subplot(2, 3, 5)
    valid_depths = depth_analysis['point_depths'][valid_mask]
    if len(valid_depths) > 0:
        ax5.hist(valid_depths, bins=30, alpha=0.7, color='green', edgecolor='black')
        ax5.axvline(np.mean(valid_depths), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {np.mean(valid_depths):.3f}m')
        ax5.set_xlabel('Depth (m)')
        ax5.set_ylabel('Number of Points')
        ax5.set_title('Depth Distribution at Valid Points')
        ax5.legend()
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, 'No valid depths', ha='center', va='center', transform=ax5.transAxes)
        ax5.set_title('Depth Distribution at Valid Points')
    
    # 6. Neighbor depth statistics
    ax6 = plt.subplot(2, 3, 6)
    neighbor_means = depth_analysis['neighbor_mean_depths']
    neighbor_stds = depth_analysis['neighbor_std_depths']
    valid_neighbor_mask = np.isfinite(neighbor_means) & np.isfinite(neighbor_stds)
    
    if np.any(valid_neighbor_mask):
        ax6.scatter(neighbor_means[valid_neighbor_mask], neighbor_stds[valid_neighbor_mask],
                   alpha=0.5, s=20, c='blue')
        ax6.set_xlabel('Mean Depth in Neighborhood (m)')
        ax6.set_ylabel('Std Depth in Neighborhood (m)')
        ax6.set_title('Depth Consistency in Neighborhood')
        ax6.grid(True, alpha=0.3)
    else:
        ax6.text(0.5, 0.5, 'No valid neighbor statistics', ha='center', va='center', transform=ax6.transAxes)
        ax6.set_title('Depth Consistency in Neighborhood')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Cannot create visualizations: frame data or depth analysis not available")

## Detailed Neighborhood Analysis

In [ ]:
# Show examples of points with different depth situations
if frame_data is not None and depth_analysis is not None:
    # Find examples: point with valid depth, point with invalid depth, point with good neighborhood, etc.
    points_2d = visible_points['points_2d']
    point_valid = depth_analysis['point_valid']
    neighbor_valid_ratios = depth_analysis['neighbor_valid_ratios']
    
    # Find interesting examples
    examples = {}
    
    # Example 1: Point with valid depth
    valid_indices = np.where(point_valid)[0]
    if len(valid_indices) > 0:
        examples['valid_point'] = valid_indices[0]
    
    # Example 2: Point with invalid depth but good neighborhood
    invalid_indices = np.where(~point_valid)[0]
    if len(invalid_indices) > 0:
        # Find one with high neighbor valid ratio
        invalid_with_good_neighbors = invalid_indices[neighbor_valid_ratios[invalid_indices] > 0.5]
        if len(invalid_with_good_neighbors) > 0:
            examples['invalid_but_good_neighbors'] = invalid_with_good_neighbors[0]
        else:
            examples['invalid_point'] = invalid_indices[0]
    
    # Example 3: Point with poor neighborhood
    poor_neighbor_indices = np.where(neighbor_valid_ratios < 0.3)[0]
    if len(poor_neighbor_indices) > 0:
        examples['poor_neighborhood'] = poor_neighbor_indices[0]
    
    # Visualize examples
    if len(examples) > 0:
        fig, axes = plt.subplots(1, len(examples), figsize=(6*len(examples), 6))
        if len(examples) == 1:
            axes = [axes]
        
        for idx, (name, point_idx) in enumerate(examples.items()):
            ax = axes[idx]
            x, y = points_2d[point_idx]
            x_int, y_int = int(np.round(x)), int(np.round(y))
            
            # Extract neighborhood
            window_size = 5
            half = window_size // 2
            H, W = frame_data['depth'].shape
            x0 = max(0, x_int - half)
            x1 = min(W, x_int + half + 1)
            y0 = max(0, y_int - half)
            y1 = min(H, y_int + half + 1)
            
            # Get RGB and depth patches
            rgb_patch = frame_data['rgb'][y0:y1, x0:x1]
            depth_patch = frame_data['depth'][y0:y1, x0:x1]
            
            # Show RGB patch
            ax.imshow(rgb_patch)
            center_x, center_y = x_int - x0, y_int - y0
            ax.scatter([center_x], [center_y], c='red', s=200, marker='x', linewidths=3)
            ax.set_title(f'{name}\\n'
                        f'Point valid: {point_valid[point_idx]}\\n'
                        f'Neighbor valid ratio: {neighbor_valid_ratios[point_idx]:.2%}\\n'
                        f'Depth at point: {depth_analysis["point_depths"][point_idx]:.3f}m' if point_valid[point_idx] else 'Depth at point: N/A')
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\\nSummary Statistics:")
        print(f"  Total visible points: {len(points_2d)}")
        print(f"  Points with valid depth: {np.sum(point_valid)} ({100*np.mean(point_valid):.1f}%)")
        print(f"  Points with invalid depth: {np.sum(~point_valid)} ({100*np.mean(~point_valid):.1f}%)")
        print(f"  Mean neighbor valid ratio: {np.mean(neighbor_valid_ratios):.2%}")
        print(f"  Points with >50% valid neighbors: {np.sum(neighbor_valid_ratios > 0.5)} ({100*np.mean(neighbor_valid_ratios > 0.5):.1f}%)")
        print(f"  Points with <30% valid neighbors: {np.sum(neighbor_valid_ratios < 0.3)} ({100*np.mean(neighbor_valid_ratios < 0.3):.1f}%)")
        
        # Depth statistics for valid points
        valid_depths = depth_analysis['point_depths'][point_valid]
        if len(valid_depths) > 0:
            print(f"\\nDepth Statistics (valid points only):")
            print(f"  Mean depth: {np.mean(valid_depths):.3f}m")
            print(f"  Std depth: {np.std(valid_depths):.3f}m")
            print(f"  Min depth: {np.min(valid_depths):.3f}m")
            print(f"  Max depth: {np.max(valid_depths):.3f}m")
    else:
        print("No examples found to visualize")
else:
    print("Cannot create detailed analysis: frame data or depth analysis not available")

## Uncertainty Visualization: RGB, Depth, and 3D Point Cloud

Visualize uncertainty overlaid on RGB and depth images, plus a 3D point cloud colored by uncertainty.

In [ ]:
# Visualize uncertainty on RGB, depth images, and as 3D point cloud
if 'uncertainty_analysis' in globals() and uncertainty_analysis is not None and frame_data is not None:
    from mpl_toolkits.mplot3d import Axes3D
    from point2pose.utils.camera import convert_pixel_to_world
    
    sigma_z = uncertainty_analysis['sigma_z']
    valid_mask = uncertainty_analysis['valid_mask']
    valid_sigma_z = uncertainty_analysis['valid_sigma_z']
    valid_points_2d = uncertainty_analysis['valid_points_2d']
    valid_depths = uncertainty_analysis['valid_depths']
    world_pts = uncertainty_analysis['world_pts']
    
    if len(valid_sigma_z) > 0:
        # Use viridis colormap like plot_registration_stats.py
        try:
            import matplotlib
            cmap_get = getattr(getattr(matplotlib, "colormaps", matplotlib.cm), "get_cmap")
            uncertainty_cmap = cmap_get("viridis")
        except (AttributeError, ImportError, KeyError):
            uncertainty_cmap = plt.cm.viridis
        
        # Normalize uncertainties (same approach as plot_registration_stats.py)
        uncertainty_min = valid_sigma_z.min()
        uncertainty_max = valid_sigma_z.max()
        uncertainty_normalized = (valid_sigma_z - uncertainty_min) / (uncertainty_max - uncertainty_min + 1e-10)
        uncertainty_colors = uncertainty_cmap(uncertainty_normalized)
        
        # Create figure with 2 subplots: RGB overlay, Depth overlay
        fig1 = plt.figure(figsize=(16, 6))
        
        # 1. RGB image with uncertainty overlay
        ax1 = plt.subplot(1, 2, 1)
        rgb_img = frame_data['rgb'].copy()
        ax1.imshow(rgb_img)
        
        # Scatter points colored by uncertainty (using viridis)
        scatter1 = ax1.scatter(valid_points_2d[:, 0], valid_points_2d[:, 1], 
                              c=uncertainty_colors, s=50, alpha=0.7,
                              edgecolors='white', linewidths=0.5)
        
        # Create colorbar with proper normalization
        sm1 = plt.cm.ScalarMappable(cmap=uncertainty_cmap, 
                                    norm=plt.Normalize(vmin=uncertainty_min, vmax=uncertainty_max))
        sm1.set_array([])
        cbar1 = plt.colorbar(sm1, ax=ax1, label='Uncertainty σ_z (m)')
        ax1.set_title('RGB Image with Uncertainty Overlay', fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        # 2. Depth image with uncertainty overlay
        ax2 = plt.subplot(1, 2, 2)
        depth_vis = frame_data['depth'].copy()
        depth_vis[depth_vis > 2.0] = 2.0
        depth_vis[depth_vis < 0.05] = 0.05
        ax2.imshow(depth_vis, cmap='gray', vmin=0.05, vmax=2.0)
        
        # Overlay uncertainty-colored points (using viridis)
        scatter2 = ax2.scatter(valid_points_2d[:, 0], valid_points_2d[:, 1], 
                               c=uncertainty_colors, s=50, alpha=0.8,
                               edgecolors='white', linewidths=0.5)
        
        # Create colorbar with proper normalization
        sm2 = plt.cm.ScalarMappable(cmap=uncertainty_cmap,
                                    norm=plt.Normalize(vmin=uncertainty_min, vmax=uncertainty_max))
        sm2.set_array([])
        cbar2 = plt.colorbar(sm2, ax=ax2, label='Uncertainty σ_z (m)')
        ax2.set_title('Depth Image with Uncertainty Overlay', fontsize=12, fontweight='bold')
        ax2.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Create separate interactive 3D point cloud plot
        # Use a backend that supports interactivity
        try:
            # Try to use widget backend for better interactivity
            import matplotlib
            if 'widget' in matplotlib.get_backend().lower() or 'notebook' in matplotlib.get_backend().lower():
                pass  # Already using interactive backend
            else:
                # Note: User may need to run %matplotlib widget in a cell before this
                print("Note: For full interactivity, run '%matplotlib widget' in a cell before this one")
        except:
            pass
        
        fig2 = plt.figure(figsize=(12, 10))
        ax3 = fig2.add_subplot(111, projection='3d')
        
        # Extract 3D points for valid uncertainty points
        valid_world_pts = world_pts[valid_mask]
        
        # Create 3D scatter plot colored by uncertainty (using viridis)
        scatter3 = ax3.scatter(valid_world_pts[:, 0], 
                               valid_world_pts[:, 1], 
                               valid_world_pts[:, 2],
                               c=uncertainty_colors, s=30, alpha=0.8,
                               edgecolors='black', linewidths=0.3)
        
        ax3.set_xlabel('X (m)', fontsize=12)
        ax3.set_ylabel('Y (m)', fontsize=12)
        ax3.set_zlabel('Z (m)', fontsize=12)
        ax3.set_title('3D Point Cloud Colored by Uncertainty\\n(Interactive - Rotate with Mouse)', 
                     fontsize=14, fontweight='bold')
        
        # Set equal aspect ratio for better visualization
        max_range = np.array([valid_world_pts[:, 0].max() - valid_world_pts[:, 0].min(),
                              valid_world_pts[:, 1].max() - valid_world_pts[:, 1].min(),
                              valid_world_pts[:, 2].max() - valid_world_pts[:, 2].min()]).max() / 2.0
        mid_x = (valid_world_pts[:, 0].max() + valid_world_pts[:, 0].min()) * 0.5
        mid_y = (valid_world_pts[:, 1].max() + valid_world_pts[:, 1].min()) * 0.5
        mid_z = (valid_world_pts[:, 2].max() + valid_world_pts[:, 2].min()) * 0.5
        ax3.set_xlim(mid_x - max_range, mid_x + max_range)
        ax3.set_ylim(mid_y - max_range, mid_y + max_range)
        ax3.set_zlim(mid_z - max_range, mid_z + max_range)
        
        # Set equal aspect ratio using set_box_aspect (if available)
        try:
            ax3.set_box_aspect([1, 1, 1])
        except AttributeError:
            pass
        
        # Add colorbar for uncertainty
        sm3 = plt.cm.ScalarMappable(cmap=uncertainty_cmap,
                                   norm=plt.Normalize(vmin=uncertainty_min, vmax=uncertainty_max))
        sm3.set_array([])
        cbar3 = fig2.colorbar(sm3, ax=ax3, pad=0.1, shrink=0.8)
        cbar3.set_label('Uncertainty σ_z (m)', rotation=270, labelpad=20, fontsize=12)
        
        # Set initial viewing angle for better visualization
        ax3.view_init(elev=20, azim=45)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary
        print(f"\\nVisualization Summary:")
        print(f"  Visible points visualized: {len(valid_sigma_z)}")
        print(f"  Uncertainty range: [{uncertainty_min:.6f}, {uncertainty_max:.6f}] m")
        print(f"  Mean uncertainty: {np.mean(valid_sigma_z):.6f} m")
        print(f"  Colormap: viridis (matching plot_registration_stats.py)")
        print(f"\\nNote: For full 3D interactivity, run '%matplotlib widget' in a cell before this one")
        
    else:
        print("No valid uncertainty values to visualize")
else:
    print("Cannot create uncertainty visualizations: uncertainty_analysis or frame_data not available")
    print("Make sure you've run the Depth Uncertainty Analysis cell first.")

## Uncertainty Computation Visualization

Visualize how depth uncertainty is computed by showing example neighborhoods around selected points.

In [ ]:
# Visualize how uncertainty is computed for example points
if 'uncertainty_analysis' in globals() and uncertainty_analysis is not None and frame_data is not None:
    window_size = 5  # Same as used in uncertainty computation
    half = window_size // 2
    H, W = frame_data['depth'].shape
    depth_image = frame_data['depth'].copy()
    depth_factor = frame_data['depth_factor']
    cam_intrinsics = frame_data['intrinsics']
    
    valid_sigma_z = uncertainty_analysis['valid_sigma_z']
    valid_points_2d = uncertainty_analysis['valid_points_2d']
    
    if len(valid_sigma_z) > 0:
        # Select a few example points with different uncertainty levels
        # Pick points at different percentiles
        percentiles = [10, 25, 50, 75, 90]
        example_indices = []
        for p in percentiles:
            target_unc = np.percentile(valid_sigma_z, p)
            idx = np.argmin(np.abs(valid_sigma_z - target_unc))
            example_indices.append(idx)
        
        # Remove duplicates
        example_indices = list(dict.fromkeys(example_indices))  # Preserves order
        num_examples = min(len(example_indices), 6)  # Show up to 6 examples
        example_indices = example_indices[:num_examples]
        
        # Create figure with subplots for each example
        n_cols = 3
        n_rows = (num_examples + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        axes = axes.flatten()
        
        for plot_idx, example_idx in enumerate(example_indices):
            ax = axes[plot_idx]
            
            pt_2d = valid_points_2d[example_idx]
            x, y = int(np.round(pt_2d[0])), int(np.round(pt_2d[1]))
            uncertainty = valid_sigma_z[example_idx]
            
            if 0 <= x < W and 0 <= y < H:
                # Extract neighborhood window
                x0 = max(0, x - half)
                x1 = min(W, x + half + 1)
                y0 = max(0, y - half)
                y1 = min(H, y + half + 1)
                
                # Get neighborhood depth values
                neighborhood = depth_image[y0:y1, x0:x1].copy()
                neighborhood_meters = neighborhood / depth_factor
                
                # Compute statistics for this neighborhood
                valid_mask = np.isfinite(neighborhood_meters) & (neighborhood_meters >= 0.05) & (neighborhood_meters <= 2.0)
                valid_depths = neighborhood_meters[valid_mask]
                
                if len(valid_depths) > 0:
                    mean_depth = np.mean(valid_depths)
                    std_depth = np.std(valid_depths)
                    var_depth = np.var(valid_depths)
                    local_sigma = np.sqrt(var_depth)
                    
                    # Display the neighborhood as a heatmap
                    im = ax.imshow(neighborhood_meters, cmap='viridis', vmin=0.05, vmax=2.0, 
                                  interpolation='nearest', aspect='auto')
                    
                    # Mark the center point
                    center_x_in_window = x - x0
                    center_y_in_window = y - y0
                    ax.plot(center_x_in_window, center_y_in_window, 'r*', markersize=15, 
                           markeredgecolor='white', markeredgewidth=1, label='Center point')
                    
                    # Add text annotations
                    ax.set_title(f'Uncertainty: {uncertainty:.6f}m\\n'
                               f'Local σ: {local_sigma:.6f}m, Mean: {mean_depth:.3f}m, Std: {std_depth:.6f}m\\n'
                               f'Valid neighbors: {len(valid_depths)}/{window_size*window_size}',
                               fontsize=10, fontweight='bold')
                    
                    # Add colorbar
                    plt.colorbar(im, ax=ax, label='Depth (m)', shrink=0.8)
                    
                    # Add grid to show window boundaries
                    ax.set_xticks(np.arange(-0.5, neighborhood_meters.shape[1], 1), minor=True)
                    ax.set_yticks(np.arange(-0.5, neighborhood_meters.shape[0], 1), minor=True)
                    ax.grid(which='minor', color='white', linestyle='-', linewidth=0.5, alpha=0.3)
                    
                    # Add text showing depth values (if window is small enough)
                    if window_size <= 7:
                        for i in range(neighborhood_meters.shape[0]):
                            for j in range(neighborhood_meters.shape[1]):
                                depth_val = neighborhood_meters[i, j]
                                if np.isfinite(depth_val) and 0.05 <= depth_val <= 2.0:
                                    ax.text(j, i, f'{depth_val:.3f}', ha='center', va='center',
                                           color='white', fontsize=7, fontweight='bold',
                                           bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))
                else:
                    ax.text(0.5, 0.5, 'Invalid neighborhood', ha='center', va='center',
                           transform=ax.transAxes, fontsize=12)
                    ax.set_title(f'Point at ({x}, {y}) - Invalid', fontsize=10)
            else:
                ax.text(0.5, 0.5, f'Point out of bounds\\n({x}, {y})', ha='center', va='center',
                       transform=ax.transAxes, fontsize=12)
                ax.set_title(f'Uncertainty: {uncertainty:.6f}m', fontsize=10)
            
            ax.axis('off')
        
        # Hide unused subplots
        for idx in range(num_examples, len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        # Print explanation
        print("\\nUncertainty Computation Explanation:")
        print("  For each point, a 5x5 window around it is extracted from the depth image.")
        print("  The uncertainty (σ_z) is computed as:")
        print("    1. Local variance: σ_local = sqrt(E[depth²] - E[depth]²) from valid neighbors")
        print("    2. Base model: σ_base = a + b * depth²")
        print("    3. Edge inflation: σ = max(σ_local, σ_base) * (1 + α * |∇depth|)")
        print("    4. Final: σ_z = clip(σ, σ_min, σ_max)")
        print("  Points with fewer valid neighbors or near depth edges have higher uncertainty.")
        
    else:
        print("No valid uncertainty values to visualize")
else:
    print("Cannot create uncertainty computation visualization: uncertainty_analysis or frame_data not available")
    print("Make sure you've run the Depth Uncertainty Analysis cell first.")

## Depth Recovery Analysis

Analyze whether we can recover depth for invalid points using neighborhood windows of different sizes.

In [ ]:
def recover_depth_from_neighborhood(
    points_2d: np.ndarray,
    depth_image: np.ndarray,
    invalid_mask: np.ndarray,
    window_sizes: list = [3, 5, 7],
    min_depth: float = 0.05,
    max_depth: float = 2.0,
    min_valid_neighbors: int = 3
) -> dict:
    """
    Attempt to recover depth for invalid points using neighborhood windows.
    
    Args:
        points_2d: (N, 2) pixel coordinates
        depth_image: (H, W) depth image
        invalid_mask: (N,) bool mask indicating which points have invalid depth
        window_sizes: List of window sizes to try (odd integers)
        min_depth: Minimum valid depth
        max_depth: Maximum valid depth
        min_valid_neighbors: Minimum number of valid neighbors required for recovery
        
    Returns:
        dict with recovery results for each window size
    """
    H, W = depth_image.shape
    invalid_indices = np.where(invalid_mask)[0]
    N_invalid = len(invalid_indices)
    
    results = {}
    
    for window_size in window_sizes:
        if window_size % 2 == 0:
            continue  # Skip even window sizes
        
        half = window_size // 2
        recovered_depths = np.full(N_invalid, np.nan)
        recovery_valid = np.zeros(N_invalid, dtype=bool)
        neighbor_counts = np.zeros(N_invalid, dtype=int)
        neighbor_stds = np.full(N_invalid, np.nan)
        
        for i, idx in enumerate(invalid_indices):
            x, y = points_2d[idx]
            x_int = int(np.round(x))
            y_int = int(np.round(y))
            
            if not (0 <= x_int < W and 0 <= y_int < H):
                continue
            
            # Extract neighborhood
            x0 = max(0, x_int - half)
            x1 = min(W, x_int + half + 1)
            y0 = max(0, y_int - half)
            y1 = min(H, y_int + half + 1)
            
            neighborhood = depth_image[y0:y1, x0:x1]
            valid_mask = np.isfinite(neighborhood) & (neighborhood >= min_depth) & (neighborhood <= max_depth)
            
            neighbor_counts[i] = np.sum(valid_mask)
            
            if np.sum(valid_mask) >= min_valid_neighbors:
                valid_depths = neighborhood[valid_mask]
                recovered_depth = np.mean(valid_depths)
                recovered_depths[i] = recovered_depth
                recovery_valid[i] = True
                neighbor_stds[i] = np.std(valid_depths)
        
        results[window_size] = {
            'recovered_depths': recovered_depths,
            'recovery_valid': recovery_valid,
            'neighbor_counts': neighbor_counts,
            'neighbor_stds': neighbor_stds,
            'recovery_rate': np.mean(recovery_valid),
            'num_recovered': np.sum(recovery_valid)
        }
    
    return results

# Perform depth recovery analysis
if frame_data is not None and depth_analysis is not None:
    invalid_mask = ~depth_analysis['point_valid']
    recovery_results = recover_depth_from_neighborhood(
        visible_points['points_2d'],
        frame_data['depth'],
        invalid_mask,
        window_sizes=[3, 5, 7],
        min_depth=0.05,
        max_depth=2.0,
        min_valid_neighbors=3
    )
    
    print("Depth Recovery Analysis:")
    print(f"  Total invalid points: {np.sum(invalid_mask)}")
    for window_size in sorted(recovery_results.keys()):
        result = recovery_results[window_size]
        print(f"  Window size {window_size}x{window_size}:")
        print(f"    Recovered: {result['num_recovered']} ({100*result['recovery_rate']:.1f}%)")
        if result['num_recovered'] > 0:
            recovered_depths = result['recovered_depths'][result['recovery_valid']]
            neighbor_stds = result['neighbor_stds'][result['recovery_valid']]
            print(f"    Mean recovered depth: {np.mean(recovered_depths):.3f}m")
            print(f"    Mean neighbor std: {np.mean(neighbor_stds):.3f}m")
            print(f"    Mean valid neighbors: {np.mean(result['neighbor_counts'][result['recovery_valid']]):.1f}")
else:
    print("Cannot perform depth recovery analysis: frame data or depth analysis not available")
    recovery_results = None

## Depth Recovery Visualizations

In [ ]:
# Visualize depth recovery results
if frame_data is not None and depth_analysis is not None and recovery_results is not None:
    fig = plt.figure(figsize=(20, 14))
    
    invalid_mask = ~depth_analysis['point_valid']
    invalid_points_2d = visible_points['points_2d'][invalid_mask]
    valid_points_2d = visible_points['points_2d'][depth_analysis['point_valid']]
    valid_depths = depth_analysis['point_depths'][depth_analysis['point_valid']]
    
    # 1. Recovery rate comparison
    ax1 = plt.subplot(3, 3, 1)
    window_sizes = sorted(recovery_results.keys())
    recovery_rates = [recovery_results[ws]['recovery_rate'] * 100 for ws in window_sizes]
    recovery_counts = [recovery_results[ws]['num_recovered'] for ws in window_sizes]
    bars = ax1.bar([f'{ws}x{ws}' for ws in window_sizes], recovery_rates, 
                   color=['skyblue', 'lightgreen', 'lightcoral'], alpha=0.7)
    ax1.set_ylabel('Recovery Rate (%)')
    ax1.set_title('Depth Recovery Rate by Window Size')
    ax1.set_ylim([0, max(recovery_rates) * 1.2 if max(recovery_rates) > 0 else 100])
    for bar, rate, count in zip(bars, recovery_rates, recovery_counts):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{rate:.1f}%\\n({count} pts)',
                ha='center', va='bottom')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. Cumulative recovery (how many can be recovered with at least one window size)
    ax2 = plt.subplot(3, 3, 2)
    # Check which points can be recovered with any window size
    any_recovery = np.zeros(len(invalid_points_2d), dtype=bool)
    for ws in window_sizes:
        any_recovery |= recovery_results[ws]['recovery_valid']
    
    recovery_status = ['Recoverable', 'Not Recoverable']
    recovery_counts_status = [np.sum(any_recovery), np.sum(~any_recovery)]
    colors_status = ['green', 'red']
    bars2 = ax2.bar(recovery_status, recovery_counts_status, color=colors_status, alpha=0.7)
    ax2.set_ylabel('Number of Points')
    ax2.set_title('Overall Recovery Potential')
    for bar, count in zip(bars2, recovery_counts_status):
        height = bar.get_height()
        total = len(invalid_points_2d)
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\\n({100*count/total:.1f}%)',
                ha='center', va='bottom')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Distribution of neighbor counts for recoverable vs non-recoverable
    ax3 = plt.subplot(3, 3, 3)
    # Use window size 7 for this analysis (largest window)
    ws7 = recovery_results[7]
    recoverable_neighbor_counts = ws7['neighbor_counts'][ws7['recovery_valid']]
    non_recoverable_neighbor_counts = ws7['neighbor_counts'][~ws7['recovery_valid']]
    
    if len(recoverable_neighbor_counts) > 0 and len(non_recoverable_neighbor_counts) > 0:
        ax3.hist([recoverable_neighbor_counts, non_recoverable_neighbor_counts], 
                bins=range(0, 50, 2), alpha=0.7, label=['Recoverable', 'Not Recoverable'],
                color=['green', 'red'], edgecolor='black')
        ax3.set_xlabel('Number of Valid Neighbors (7x7 window)')
        ax3.set_ylabel('Number of Points')
        ax3.set_title('Neighbor Count Distribution')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    # 4-6. Recovered depth distributions for each window size
    for idx, ws in enumerate(window_sizes):
        ax = plt.subplot(3, 3, 4 + idx)
        result = recovery_results[ws]
        recovered_depths = result['recovered_depths'][result['recovery_valid']]
        
        if len(recovered_depths) > 0:
            ax.hist(recovered_depths, bins=30, alpha=0.7, color='blue', edgecolor='black')
            ax.axvline(np.mean(recovered_depths), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {np.mean(recovered_depths):.3f}m')
            ax.set_xlabel('Recovered Depth (m)')
            ax.set_ylabel('Number of Points')
            ax.set_title(f'Recovered Depth Distribution ({ws}x{ws} window)\\n{len(recovered_depths)} points recovered')
            ax.legend()
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'No points recovered\\nwith {ws}x{ws} window', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Recovered Depth Distribution ({ws}x{ws} window)')
    
    # 7. Comparison: Valid depths vs Recovered depths
    ax7 = plt.subplot(3, 3, 7)
    if len(valid_depths) > 0:
        # Get recovered depths from largest window
        ws7_recovered = recovery_results[7]['recovered_depths'][recovery_results[7]['recovery_valid']]
        if len(ws7_recovered) > 0:
            ax7.hist([valid_depths, ws7_recovered], bins=30, alpha=0.7, 
                    label=['Original Valid', 'Recovered (7x7)'], 
                    color=['green', 'blue'], edgecolor='black')
            ax7.set_xlabel('Depth (m)')
            ax7.set_ylabel('Number of Points')
            ax7.set_title('Depth Distribution Comparison')
            ax7.legend()
            ax7.grid(True, alpha=0.3)
        else:
            ax7.hist(valid_depths, bins=30, alpha=0.7, color='green', edgecolor='black', label='Original Valid')
            ax7.set_xlabel('Depth (m)')
            ax7.set_ylabel('Number of Points')
            ax7.set_title('Depth Distribution (No Recovery)')
            ax7.legend()
            ax7.grid(True, alpha=0.3)
    
    # 8. Neighbor consistency (std of neighbors) for recovered points
    ax8 = plt.subplot(3, 3, 8)
    all_stds = []
    all_labels = []
    for ws in window_sizes:
        result = recovery_results[ws]
        recovered_stds = result['neighbor_stds'][result['recovery_valid']]
        if len(recovered_stds) > 0:
            all_stds.append(recovered_stds)
            all_labels.append(f'{ws}x{ws}')
    
    if len(all_stds) > 0:
        ax8.boxplot(all_stds, labels=all_labels)
        ax8.set_ylabel('Std of Neighbor Depths (m)')
        ax8.set_title('Depth Consistency in Neighborhoods')
        ax8.grid(True, alpha=0.3, axis='y')
    
    # 9. Spatial visualization: Invalid points colored by recoverability
    ax9 = plt.subplot(3, 3, 9)
    ax9.imshow(frame_data['rgb'])
    # Use window size 7 for recoverability
    ws7_recoverable = recovery_results[7]['recovery_valid']
    recoverable_pts = invalid_points_2d[ws7_recoverable]
    non_recoverable_pts = invalid_points_2d[~ws7_recoverable]
    
    if len(recoverable_pts) > 0:
        ax9.scatter(recoverable_pts[:, 0], recoverable_pts[:, 1], 
                   c='yellow', s=30, alpha=0.6, label=f'Recoverable ({len(recoverable_pts)})',
                   edgecolors='orange', linewidths=0.5)
    if len(non_recoverable_pts) > 0:
        ax9.scatter(non_recoverable_pts[:, 0], non_recoverable_pts[:, 1], 
                   c='red', s=30, alpha=0.6, label=f'Not Recoverable ({len(non_recoverable_pts)})',
                   edgecolors='darkred', linewidths=0.5)
    if len(valid_points_2d) > 0:
        ax9.scatter(valid_points_2d[:, 0], valid_points_2d[:, 1], 
                   c='green', s=20, alpha=0.4, label=f'Valid ({len(valid_points_2d)})',
                   edgecolors='darkgreen', linewidths=0.3)
    ax9.set_title('Spatial Distribution: Recovery Potential')
    ax9.legend(fontsize=8)
    ax9.axis('off')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Cannot create recovery visualizations: required data not available")

## Reasonableness Analysis of Recovered Depths

In [ ]:
def evaluate_recovery_reasonableness(
    points_2d: np.ndarray,
    depth_image: np.ndarray,
    invalid_mask: np.ndarray,
    recovery_results: dict,
    valid_points_2d: np.ndarray,
    valid_depths: np.ndarray,
    window_size: int = 7,
    max_spatial_distance: float = 20.0,  # pixels
    max_depth_diff: float = 0.1  # meters
) -> dict:
    """
    Evaluate how reasonable recovered depths are by comparing with nearby valid points.
    
    Args:
        points_2d: All point coordinates
        depth_image: Depth image
        invalid_mask: Mask for invalid points
        recovery_results: Results from recover_depth_from_neighborhood
        valid_points_2d: Coordinates of valid points
        valid_depths: Depths of valid points
        window_size: Window size to use for recovery
        max_spatial_distance: Maximum pixel distance to consider for comparison
        max_depth_diff: Maximum depth difference to consider "reasonable"
        
    Returns:
        dict with reasonableness metrics
    """
    from scipy.spatial.distance import cdist
    
    invalid_indices = np.where(invalid_mask)[0]
    invalid_points = points_2d[invalid_indices]
    
    if window_size not in recovery_results:
        return {}
    
    result = recovery_results[window_size]
    recovered_mask = result['recovery_valid']
    recovered_depths = result['recovered_depths']
    
    # Find recovered points
    recovered_points = invalid_points[recovered_mask]
    recovered_depths_valid = recovered_depths[recovered_mask]
    
    if len(recovered_points) == 0 or len(valid_points_2d) == 0:
        return {
            'num_recovered': 0,
            'num_with_nearby_valid': 0,
            'num_reasonable': 0,
            'mean_depth_diff': np.nan,
            'mean_spatial_distance': np.nan
        }
    
    # Compute distances from recovered points to valid points
    distances = cdist(recovered_points, valid_points_2d)
    
    # For each recovered point, find nearest valid point
    nearest_valid_idx = np.argmin(distances, axis=1)
    nearest_distances = distances[np.arange(len(recovered_points)), nearest_valid_idx]
    nearest_valid_depths = valid_depths[nearest_valid_idx]
    
    # Compute depth differences
    depth_diffs = np.abs(recovered_depths_valid - nearest_valid_depths)
    
    # Reasonable if: nearby (within max_spatial_distance) AND similar depth (within max_depth_diff)
    nearby_mask = nearest_distances <= max_spatial_distance
    similar_depth_mask = depth_diffs <= max_depth_diff
    reasonable_mask = nearby_mask & similar_depth_mask
    
    return {
        'num_recovered': len(recovered_points),
        'num_with_nearby_valid': np.sum(nearby_mask),
        'num_reasonable': np.sum(reasonable_mask),
        'mean_depth_diff': np.mean(depth_diffs),
        'mean_spatial_distance': np.mean(nearest_distances),
        'depth_diffs': depth_diffs,
        'spatial_distances': nearest_distances,
        'reasonable_mask': reasonable_mask,
        'nearby_mask': nearby_mask,
        'similar_depth_mask': similar_depth_mask
    }

# Evaluate reasonableness
if frame_data is not None and depth_analysis is not None and recovery_results is not None:
    valid_mask = depth_analysis['point_valid']
    valid_points_2d = visible_points['points_2d'][valid_mask]
    valid_depths = depth_analysis['point_depths'][valid_mask]
    
    reasonableness_results = {}
    for ws in [3, 5, 7]:
        if ws in recovery_results:
            reasonableness_results[ws] = evaluate_recovery_reasonableness(
                visible_points['points_2d'],
                frame_data['depth'],
                ~depth_analysis['point_valid'],
                recovery_results,
                valid_points_2d,
                valid_depths,
                window_size=ws,
                max_spatial_distance=20.0,
                max_depth_diff=0.1
            )
    
    print("\\nReasonableness Analysis:")
    print("(Comparing recovered depths with nearby valid points)")
    for ws in sorted(reasonableness_results.keys()):
        res = reasonableness_results[ws]
        if res['num_recovered'] > 0:
            print(f"\\n  Window size {ws}x{ws}:")
            print(f"    Recovered points: {res['num_recovered']}")
            print(f"    With nearby valid points (<20px): {res['num_with_nearby_valid']} ({100*res['num_with_nearby_valid']/res['num_recovered']:.1f}%)")
            print(f"    Reasonable (nearby + similar depth): {res['num_reasonable']} ({100*res['num_reasonable']/res['num_recovered']:.1f}%)")
            print(f"    Mean depth difference: {res['mean_depth_diff']:.4f}m")
            print(f"    Mean spatial distance: {res['mean_spatial_distance']:.2f}px")
        else:
            print(f"\\n  Window size {ws}x{ws}: No points recovered")
else:
    print("Cannot evaluate reasonableness: required data not available")
    reasonableness_results = None

## Reasonableness Visualizations

In [ ]:
# Visualize reasonableness analysis
if frame_data is not None and depth_analysis is not None and recovery_results is not None and reasonableness_results is not None:
    fig = plt.figure(figsize=(18, 12))
    
    # 1. Reasonableness rate by window size
    ax1 = plt.subplot(2, 3, 1)
    window_sizes = sorted([ws for ws in reasonableness_results.keys() if reasonableness_results[ws]['num_recovered'] > 0])
    if len(window_sizes) > 0:
        reasonable_rates = []
        reasonable_counts = []
        for ws in window_sizes:
            res = reasonableness_results[ws]
            if res['num_recovered'] > 0:
                rate = 100 * res['num_reasonable'] / res['num_recovered']
                reasonable_rates.append(rate)
                reasonable_counts.append(res['num_reasonable'])
            else:
                reasonable_rates.append(0)
                reasonable_counts.append(0)
        
        bars = ax1.bar([f'{ws}x{ws}' for ws in window_sizes], reasonable_rates,
                      color=['skyblue', 'lightgreen', 'lightcoral'], alpha=0.7)
        ax1.set_ylabel('Reasonableness Rate (%)')
        ax1.set_title('Recovery Reasonableness by Window Size\\n(Nearby + Similar Depth)')
        ax1.set_ylim([0, max(reasonable_rates) * 1.2 if max(reasonable_rates) > 0 else 100])
        for bar, rate, count in zip(bars, reasonable_rates, reasonable_counts):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{rate:.1f}%\\n({count} pts)',
                    ha='center', va='bottom')
        ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. Depth difference distribution
    ax2 = plt.subplot(2, 3, 2)
    # Use largest window for detailed analysis
    ws7_res = reasonableness_results.get(7)
    if ws7_res and ws7_res['num_recovered'] > 0 and len(ws7_res['depth_diffs']) > 0:
        depth_diffs = ws7_res['depth_diffs']
        ax2.hist(depth_diffs, bins=30, alpha=0.7, color='blue', edgecolor='black')
        ax2.axvline(0.1, color='red', linestyle='--', linewidth=2, 
                   label='Reasonableness threshold (0.1m)')
        ax2.axvline(np.mean(depth_diffs), color='green', linestyle='--', linewidth=2,
                   label=f'Mean: {np.mean(depth_diffs):.4f}m')
        ax2.set_xlabel('Depth Difference from Nearest Valid Point (m)')
        ax2.set_ylabel('Number of Points')
        ax2.set_title('Depth Difference Distribution (7x7 window)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # 3. Spatial distance distribution
    ax3 = plt.subplot(2, 3, 3)
    if ws7_res and ws7_res['num_recovered'] > 0 and len(ws7_res['spatial_distances']) > 0:
        spatial_dists = ws7_res['spatial_distances']
        ax3.hist(spatial_dists, bins=30, alpha=0.7, color='purple', edgecolor='black')
        ax3.axvline(20.0, color='red', linestyle='--', linewidth=2,
                    label='Nearby threshold (20px)')
        ax3.axvline(np.mean(spatial_dists), color='green', linestyle='--', linewidth=2,
                   label=f'Mean: {np.mean(spatial_dists):.2f}px')
        ax3.set_xlabel('Distance to Nearest Valid Point (pixels)')
        ax3.set_ylabel('Number of Points')
        ax3.set_title('Spatial Distance Distribution (7x7 window)')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    # 4. Scatter: Depth difference vs Spatial distance
    ax4 = plt.subplot(2, 3, 4)
    if ws7_res and ws7_res['num_recovered'] > 0:
        spatial_dists = ws7_res['spatial_distances']
        depth_diffs = ws7_res['depth_diffs']
        reasonable_mask = ws7_res['reasonable_mask']
        
        if np.any(reasonable_mask):
            ax4.scatter(spatial_dists[reasonable_mask], depth_diffs[reasonable_mask],
                       c='green', s=30, alpha=0.6, label='Reasonable', edgecolors='darkgreen', linewidths=0.5)
        if np.any(~reasonable_mask):
            ax4.scatter(spatial_dists[~reasonable_mask], depth_diffs[~reasonable_mask],
                       c='red', s=30, alpha=0.6, label='Not Reasonable', edgecolors='darkred', linewidths=0.5)
        
        ax4.axhline(0.1, color='red', linestyle='--', linewidth=1, alpha=0.5)
        ax4.axvline(20.0, color='red', linestyle='--', linewidth=1, alpha=0.5)
        ax4.set_xlabel('Spatial Distance (pixels)')
        ax4.set_ylabel('Depth Difference (m)')
        ax4.set_title('Recovery Reasonableness Scatter')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    # 5. Breakdown: Reasonable vs Not Reasonable
    ax5 = plt.subplot(2, 3, 5)
    if ws7_res and ws7_res['num_recovered'] > 0:
        reasonable_count = ws7_res['num_reasonable']
        not_reasonable_count = ws7_res['num_recovered'] - reasonable_count
        
        categories = ['Reasonable', 'Not Reasonable']
        counts = [reasonable_count, not_reasonable_count]
        colors = ['green', 'red']
        bars = ax5.bar(categories, counts, color=colors, alpha=0.7)
        ax5.set_ylabel('Number of Points')
        ax5.set_title('Reasonableness Breakdown (7x7 window)')
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            total = ws7_res['num_recovered']
            ax5.text(bar.get_x() + bar.get_width()/2., height,
                    f'{count}\\n({100*count/total:.1f}%)',
                    ha='center', va='bottom')
        ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. Summary statistics table
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')
    
    # Create summary table
    summary_data = []
    summary_data.append(['Metric', 'Window 3x3', 'Window 5x5', 'Window 7x7'])
    
    if 3 in recovery_results:
        rec3 = recovery_results[3]
        summary_data.append(['Recovery Rate', f"{100*rec3['recovery_rate']:.1f}%", '', ''])
    if 5 in recovery_results:
        rec5 = recovery_results[5]
        summary_data[1][2] = f"{100*rec5['recovery_rate']:.1f}%"
    if 7 in recovery_results:
        rec7 = recovery_results[7]
        summary_data[1][3] = f"{100*rec7['recovery_rate']:.1f}%"
    
    if 3 in reasonableness_results and reasonableness_results[3]['num_recovered'] > 0:
        res3 = reasonableness_results[3]
        summary_data.append(['Reasonableness', f"{100*res3['num_reasonable']/res3['num_recovered']:.1f}%", '', ''])
    if 5 in reasonableness_results and reasonableness_results[5]['num_recovered'] > 0:
        res5 = reasonableness_results[5]
        summary_data[2][2] = f"{100*res5['num_reasonable']/res5['num_recovered']:.1f}%"
    if 7 in reasonableness_results and reasonableness_results[7]['num_recovered'] > 0:
        res7 = reasonableness_results[7]
        summary_data[2][3] = f"{100*res7['num_reasonable']/res7['num_recovered']:.1f}%"
    
    table = ax6.table(cellText=summary_data, cellLoc='center', loc='center',
                     colWidths=[0.4, 0.2, 0.2, 0.2])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    ax6.set_title('Summary Statistics', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed summary
    print("\\n" + "="*60)
    print("DEPTH RECOVERY SUMMARY")
    print("="*60)
    invalid_count = np.sum(~depth_analysis['point_valid'])
    print(f"\\nTotal invalid points: {invalid_count}")
    
    for ws in sorted(recovery_results.keys()):
        rec = recovery_results[ws]
        res = reasonableness_results.get(ws, {})
        print(f"\\nWindow {ws}x{ws}:")
        print(f"  Recovered: {rec['num_recovered']} ({100*rec['recovery_rate']:.1f}%)")
        if res.get('num_recovered', 0) > 0:
            print(f"  Reasonable: {res['num_reasonable']} ({100*res['num_reasonable']/res['num_recovered']:.1f}%)")
            print(f"  Mean depth diff: {res['mean_depth_diff']:.4f}m")
            print(f"  Mean spatial dist: {res['mean_spatial_distance']:.2f}px")
    
    # Overall assessment
    ws7_rec = recovery_results[7]
    ws7_res = reasonableness_results.get(7, {})
    if ws7_rec['num_recovered'] > 0:
        recovery_potential = 100 * ws7_rec['num_recovered'] / invalid_count
        if ws7_res.get('num_recovered', 0) > 0:
            reasonable_potential = 100 * ws7_res['num_reasonable'] / ws7_rec['num_recovered']
        else:
            reasonable_potential = 0
        
        print(f"\\nOverall Assessment (7x7 window):")
        print(f"  Can recover {recovery_potential:.1f}% of invalid points")
        print(f"  Of recovered points, {reasonable_potential:.1f}% are reasonable")
        print(f"  Overall reasonable recovery: {recovery_potential * reasonable_potential / 100:.1f}% of invalid points")
        
else:
    print("Cannot create reasonableness visualizations: required data not available")